# Latent class modeling

Use PyMC to specify a Bayesian latent class model.

In [ ]:
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pymc as pm

from utils import check_save_path, plot_corr

# Hardcoded RNG object for this script for reproducibility
RANDOM_SEED = 8066
rng = np.random.default_rng(RANDOM_SEED)


# Hardcoded path, change to apply to new data
data_path = "data/preprocessed/esa_2018.csv"
vars_info_path = "data/preprocessed/esa_grouped_vars_info.csv"
outputs_folder = "outputs"
allow_overwrite = True

In [ ]:
# Load data and information about variables
data = pd.read_csv(data_path, index_col=0, low_memory=False)
vars_info = pd.read_csv(vars_info_path, index_col=0, low_memory=False)

# Define variables
indicator_vars = (
    pd.Series(vars_info.index[vars_info["type"] == "categorical"])  # change which variables are included later
    .sample(n=20, random_state=np.random.RandomState(rng.bit_generator))  # reduce size for now
    .values
)

training_data = (
    data.loc[:, indicator_vars].sample(
        n=1000, axis=0, random_state=np.random.RandomState(rng.bit_generator)
    )  # reduce size for now
)
num_respondents, num_questions = training_data.shape  # N, J
num_choices_list = [val.max() + 1 for var, val in training_data.items()]  # A

# Plot the initial correlation matrix
plot_corr(training_data, var_names=indicator_vars, tick_label_size=9)
plt.show()

## Specify the Bayesian model

With $K$ latent classes, $J$ questions, where question $j$ has $A_j$ possible answers:

$\displaystyle \Gamma \sim \mathrm{Dirichlet}\Big(\frac{\alpha}{K} \, \mathbf{1}_K\Big) \quad \big(\text{sparse exchangeable Dirichlet class probability prior}\big)$\
$C \mid \Gamma = \gamma \sim \mathrm{Categorical}(\gamma) \quad \big(\mathrm{supp}(C) \in \{0, \dotsc, K - 1\}, C \text{ is class membership}\big)$

$\Pi_j \mid C = c \sim \mathrm{Dirichlet}(\beta \, \mathbf{1}_{A_j}) \quad \forall j \in \{0, \dotsc, J - 1\} \quad \big(\text{answer distribution for question } j \text{ is different in every latent class}\big)$\
$X_j \mid \Pi_j = \pi_j \sim \mathrm{Categorical}(\pi_j) \quad \forall j \in \{0, \dotsc, J - 1\} \quad \big(\text{answer drawn according to } \pi_j \text{ implicitly depends on } c\big)$

In [ ]:
num_latent = 50
alpha = 1.0  # sparsity parameter for classes
beta = 1.0  # sparsity parameter for answers

with pm.Model() as model:
    # Prior consumer type probabilities (exchangeable Dirichlet distribution)
    gamma = pm.Dirichlet("gamma", a=np.full(shape=(num_latent,), fill_value=alpha / num_latent), shape=(num_latent,))

    # Consumer types of respondents conditional on consumer type probabilities
    c = pm.Categorical("c", p=gamma, shape=(num_respondents,))

    # Prior answer probabilities conditional on consumer types (exchangeable Dirichlet prior)
    pi = []
    pi_c = []
    for j, num_choices in zip(range(num_questions), num_choices_list, strict=True):
        pi_j = pm.Dirichlet(
            f"pi_{j}", a=np.full(shape=(num_choices,), fill_value=beta / num_choices), shape=(num_latent, num_choices)
        )
        pi_c_j = pm.Deterministic(f"pi(c)_{j}", pi_j[c])  # insert deterministic to correct variable dependency
        pi.append(pi_j)
        pi_c.append(pi_c_j)

    # Observed answers to questions given answer probabilities
    x = []
    for j, pi_c_j, (qu_j_name, qu_j) in zip(range(num_questions), pi_c, training_data.items(), strict=True):
        x_j = pm.Categorical(f"x_{j}", p=pi_c_j, observed=qu_j, shape=(num_respondents,))
        x.append(x_j)

## Check model well-specification

In [ ]:
# Check likelihood is strictly positive and not zero
model.debug(verbose=True)

## Model information

In [ ]:
# The specified random variables
model.basic_RVs

In [ ]:
# Creates and saves graph of model (needs installation of graphviz)
graphviz_digraph = model.to_graphviz()

save_path = os.path.join(outputs_folder, "graphviz.svg")
save_path_name, save_path_ending = os.path.splitext(save_path)
save_path_ending = save_path_ending[1:]
check_save_path(save_path, allow_overwrite=allow_overwrite)
graphviz_digraph.render(filename=save_path_name, format=save_path_ending, cleanup=True)
print(f"Saved Bayesian graph visualized with graphviz to {save_path}.")

graphviz_digraph

In [ ]:
# Creates and saves graph of model
networkx_digraph = pm.model_to_networkx(model)
step = 0.1
pos = (
    {"gamma": (0, 4), "c": (0, 3)}
    | {f"pi_{j}": ((j - num_questions / 2) * step, 2) for j in range(num_questions)}
    | {f"pi(c)_{j}": ((j - num_questions / 2) * step, 1) for j in range(num_questions)}
    | {f"x_{j}": ((j - num_questions / 2) * step, 0) for j in range(num_questions)}
)

plt.figure(figsize=(6, 4))
nx.draw(networkx_digraph, pos, with_labels=True, arrows=True, node_size=300, node_color="lightblue")
plt.title("Directed Graph")

save_path = os.path.join(outputs_folder, "networkx.svg")
# save_path_name, save_path_ending = os.path.splitext(save_path)
# save_path_ending = save_path_ending[1:]
check_save_path(save_path, allow_overwrite=allow_overwrite)
plt.savefig(save_path, dpi=300)
print(f"Saved Bayesian graph visualized with networkx to {save_path}.")

plt.show()

## Sample prior predictive distribution

In [ ]:
# Sample prior predictive distribution
with model:
    prior_pred_samples = pm.sample_prior_predictive(draws=100)
prior_pred_samples

## Sample modeled distribution

In [ ]:
# Sample distribution using HMC
with model:
    trace = pm.sample(
        draws=100,
        tune=100,
        progressbar=True,
    )

In [ ]:
# Plot marginal distribution of c and gamma
pm.plot_trace(trace, var_names=["gamma", "c"])
plt.show()

In [ ]:
# Plot energy distribution from running the HMC algorithm
pm.plot_energy(trace)
plt.show()

## Obtain samples from posterior predictive distribution

In [ ]:
# Obtain samples from posterior predictive distribution
with model:
    post_pred_samples = pm.sample_posterior_predictive(trace)
post_pred_samples

In [ ]:
# Plot posterior distribution of c and gamma
pm.plot_posterior(trace, var_names=["gamma", "c"])
plt.show()